# SurfaceTarget-CRC — phase 01 — patch 1 13

Extracted from the original Google Colab research notebook.

**Note:** Earlier patches and project data may be required to execute these cells.


In [ ]:
# ============================================================
# SurfaceTarget-CRC
# Phase 1 — Patch 1.13
#
# RNA-SEQ QUALITY CROSS-VALIDATION
#
# Objectives:
# 1. Examine all three STAR counting modes
# 2. Investigate gene assignment statistics
# 3. Evaluate protein-coding gene detection
# 4. Verify gene annotation consistency
# 5. Calculate library-normalized expression correlations
#
# IMPORTANT:
# This is a diagnostic assessment.
# No final RNA-seq file selection is performed.
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from itertools import combinations

import pandas as pd
import numpy as np
import hashlib
import json

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

PROJECT_ROOT = Path("/content/SurfaceTarget-CRC")

METADATA_DIR = PROJECT_ROOT / "data" / "metadata"

RAW_DIR = (
    PROJECT_ROOT /
    "data/raw/tcga_duplicate_qc"
)

RESULTS_DIR = PROJECT_ROOT / "results/tables"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

AUDIT_PATH = (
    METADATA_DIR /
    "tcga_duplicate_rnaseq_provenance_audit.csv"
)

COUNT_COLUMNS = [
    "unstranded",
    "stranded_first",
    "stranded_second"
]

STAR_SUMMARY_ROWS = [
    "N_unmapped",
    "N_multimapping",
    "N_noFeature",
    "N_ambiguous"
]

print("=" * 65)
print("SURFACETARGET-CRC")
print("PATCH 1.13 — RNA-SEQ QUALITY CROSS-VALIDATION")
print("=" * 65)

# ------------------------------------------------------------
# 2. Load metadata
# ------------------------------------------------------------

audit = pd.read_csv(
    AUDIT_PATH,
    dtype=str,
    keep_default_na=False
)

if audit["file_id"].duplicated().any():
    raise RuntimeError(
        "Duplicate file IDs in provenance audit."
    )

print("\nPatients:", audit["patient_id"].nunique())
print("Files:", len(audit))

# ------------------------------------------------------------
# 3. Load and validate STAR count files
# ------------------------------------------------------------

def load_star_file(file_path):

    df = pd.read_csv(
        file_path,
        sep="\t",
        comment="#",
        low_memory=False,
        dtype={"gene_id": str}
    )

    required = {
        "gene_id",
        *COUNT_COLUMNS
    }

    missing = required - set(df.columns)

    if missing:
        raise RuntimeError(
            f"Missing STAR columns: {missing}"
        )

    summary = df[
        df["gene_id"].isin(STAR_SUMMARY_ROWS)
    ].copy()

    genes = df[
        ~df["gene_id"].isin(STAR_SUMMARY_ROWS)
    ].copy()

    if genes["gene_id"].duplicated().any():
        raise RuntimeError(
            "Duplicate gene identifiers."
        )

    if set(summary["gene_id"]) != set(STAR_SUMMARY_ROWS):
        raise RuntimeError(
            "Incomplete STAR count-summary rows."
        )

    for column in COUNT_COLUMNS:

        genes[column] = pd.to_numeric(
            genes[column],
            errors="raise"
        )

        summary[column] = pd.to_numeric(
            summary[column],
            errors="raise"
        )

        if genes[column].isna().any():
            raise RuntimeError(
                f"Missing counts: {column}"
            )

        if (genes[column] < 0).any():
            raise RuntimeError(
                f"Negative counts: {column}"
            )

    return genes, summary


# ------------------------------------------------------------
# 4. Examine counting statistics
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("STAR COUNTING MODE ASSESSMENT")
print("-" * 65)

qc_records = []

gene_profiles = {}

gene_id_signatures = {}

for _, row in audit.iterrows():

    file_id = row["file_id"]

    file_path = RAW_DIR / f"{file_id}.tsv"

    if not file_path.exists():
        raise FileNotFoundError(
            f"Missing downloaded file: {file_path}"
        )

    genes, summary = load_star_file(file_path)

    # Verify gene identifier structure.

    gene_ids = sorted(
        genes["gene_id"].astype(str).tolist()
    )

    signature = hashlib.sha256(
        "\n".join(gene_ids).encode("utf-8")
    ).hexdigest()

    gene_id_signatures[file_id] = signature

    # Store unstranded profiles.

    profile = genes.set_index(
        "gene_id"
    )["unstranded"]

    gene_profiles[file_id] = profile

    # Identify protein-coding genes.

    if "gene_type" in genes.columns:

        protein_coding = genes[
            genes["gene_type"] == "protein_coding"
        ].copy()

    else:

        protein_coding = pd.DataFrame()

    for column in COUNT_COLUMNS:

        assigned = float(
            genes[column].sum()
        )

        star_stats = (
            summary.set_index("gene_id")[column]
            .to_dict()
        )

        unmapped = float(
            star_stats["N_unmapped"]
        )

        multimapping = float(
            star_stats["N_multimapping"]
        )

        no_feature = float(
            star_stats["N_noFeature"]
        )

        ambiguous = float(
            star_stats["N_ambiguous"]
        )

        total = (
            assigned
            + unmapped
            + multimapping
            + no_feature
            + ambiguous
        )

        if total <= 0:
            raise RuntimeError(
                f"Invalid total counts: {file_id}"
            )

        assignment_pct = (
            100 * assigned / total
        )

        ambiguous_pct = (
            100 * ambiguous / total
        )

        no_feature_pct = (
            100 * no_feature / total
        )

        detected_genes = int(
            (genes[column] > 0).sum()
        )

        if not protein_coding.empty:

            detected_protein_coding = int(
                (protein_coding[column] > 0).sum()
            )

            total_protein_coding = len(
                protein_coding
            )

        else:

            detected_protein_coding = np.nan
            total_protein_coding = np.nan

        qc_records.append({

            "patient_id": row["patient_id"],

            "file_id": file_id,

            "file_short": file_id[:8],

            "sample_id": row["sample_ids"],

            "count_mode": column,

            "total_genes": len(genes),

            "detected_genes": detected_genes,

            "total_protein_coding": total_protein_coding,

            "detected_protein_coding":
                detected_protein_coding,

            "assigned_counts": assigned,

            "assignment_percent": assignment_pct,

            "ambiguous_percent": ambiguous_pct,

            "no_feature_percent": no_feature_pct,

            "unmapped_counts": unmapped,

            "multimapping_counts": multimapping
        })

qc = pd.DataFrame(qc_records)

# ------------------------------------------------------------
# 5. Display count-mode comparison
# ------------------------------------------------------------

print("\nUnstranded and stranded count statistics:")

display_columns = [
    "patient_id",
    "file_short",
    "sample_id",
    "count_mode",
    "detected_genes",
    "detected_protein_coding",
    "assignment_percent",
    "ambiguous_percent",
    "no_feature_percent"
]

print(
    qc[display_columns]
    .round(2)
    .to_string(index=False)
)

# ------------------------------------------------------------
# 6. Verify gene identifier consistency
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("GENE ANNOTATION CONSISTENCY")
print("-" * 65)

unique_signatures = set(
    gene_id_signatures.values()
)

print(
    "\nUnique gene identifier signatures:",
    len(unique_signatures)
)

print("\nGene counts by file:")

for file_id, profile in gene_profiles.items():

    print(
        f"{file_id[:8]}: "
        f"{len(profile)} genes"
    )

if len(unique_signatures) == 1:

    print("\nGene identifier consistency: PASSED")

else:

    print(
        "\nWARNING: Gene identifier sets differ."
    )

    print(
        "Files may require additional annotation "
        "harmonization before comparison."
    )

# ------------------------------------------------------------
# 7. Protein-coding gene profiles
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("NORMALIZED EXPRESSION CORRELATIONS")
print("-" * 65)

# We use log2(CPM + 1) for descriptive QC.
# This is NOT the normalization used for
# downstream differential expression.

normalized_profiles = {}

protein_coding_id_sets = {}

for _, row in audit.iterrows():

    file_id = row["file_id"]

    file_path = RAW_DIR / f"{file_id}.tsv"

    genes, _ = load_star_file(file_path)

    if "gene_type" not in genes.columns:

        raise RuntimeError(
            "gene_type column unavailable. "
            "Protein-coding correlation cannot proceed."
        )

    coding = genes[
        genes["gene_type"] == "protein_coding"
    ].copy()

    if coding.empty:
        raise RuntimeError(
            "No protein-coding annotations found."
        )

    protein_coding_id_sets[file_id] = set(
        coding["gene_id"]
    )

    counts = (
        coding.set_index("gene_id")["unstranded"]
        .astype(float)
    )

    # Use total counts across all annotated genes
    # as the descriptive CPM denominator.

    total_assigned = float(
        genes["unstranded"].sum()
    )

    if total_assigned <= 0:
        raise RuntimeError(
            f"Zero assigned counts: {file_id}"
        )

    cpm = (
        counts / total_assigned
    ) * 1_000_000

    normalized_profiles[file_id] = np.log2(
        cpm + 1
    )

# ------------------------------------------------------------
# 8. Compare protein-coding expression
# ------------------------------------------------------------

correlation_records = []

for patient_id, group in audit.groupby("patient_id"):

    print("\nPatient:", patient_id)

    file_ids = group["file_id"].tolist()

    patient_profiles = pd.concat(
        [
            normalized_profiles[fid].rename(fid)
            for fid in file_ids
        ],
        axis=1,
        join="inner"
    )

    if patient_profiles.empty:
        raise RuntimeError(
            "No shared protein-coding genes."
        )

    # Pearson correlation on normalized profiles.

    correlation_matrix = (
        patient_profiles.corr(method="pearson")
    )

    short_names = {
        fid: fid[:8]
        for fid in file_ids
    }

    display_matrix = correlation_matrix.rename(
        index=short_names,
        columns=short_names
    )

    print("\nPearson correlation: log2(CPM + 1)")

    print(
        display_matrix.round(4).to_string()
    )

    # Save pairwise correlations.

    for file_a, file_b in combinations(file_ids, 2):

        correlation_records.append({

            "patient_id": patient_id,

            "file_a": file_a,

            "file_b": file_b,

            "pearson_log2_cpm": float(
                correlation_matrix.loc[
                    file_a,
                    file_b
                ]
            ),

            "shared_protein_coding_genes":
                len(patient_profiles)
        })

correlations = pd.DataFrame(
    correlation_records
)

# ------------------------------------------------------------
# 9. Display interpretation notes
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("QUALITY INTERPRETATION")
print("-" * 65)

print(
    "\n1. Assignment fraction is not "
    "the same as genome alignment rate."
)

print(
    "2. Gene detection depends on sequencing "
    "depth and library characteristics."
)

print(
    "3. Protein-coding gene detection can provide "
    "additional context for unusual samples."
)

print(
    "4. Normalized correlation is descriptive "
    "and is not an independent quality score."
)

print(
    "5. No sample has been automatically "
    "excluded based on these metrics."
)

# ------------------------------------------------------------
# 10. Save results
# ------------------------------------------------------------

qc_path = (
    RESULTS_DIR /
    "tcga_duplicate_rnaseq_extended_qc.csv"
)

correlation_path = (
    RESULTS_DIR /
    "tcga_duplicate_normalized_correlations.csv"
)

signature_path = (
    METADATA_DIR /
    "tcga_duplicate_gene_identifier_signatures.json"
)

qc.to_csv(
    qc_path,
    index=False
)

correlations.to_csv(
    correlation_path,
    index=False
)

with open(signature_path, "w") as f:

    json.dump(
        gene_id_signatures,
        f,
        indent=4
    )

# ------------------------------------------------------------
# 11. Save provenance
# ------------------------------------------------------------

provenance = {

    "project": "SurfaceTarget-CRC",

    "patch": "1.13",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "files_assessed": int(
        audit["file_id"].nunique()
    ),

    "count_modes": COUNT_COLUMNS,

    "gene_identifier_consistent": (
        len(unique_signatures) == 1
    ),

    "correlation_method": (
        "Pearson correlation of log2(CPM + 1) "
        "protein-coding gene expression"
    ),

    "cpm_denominator": (
        "Total unstranded counts assigned "
        "to all annotated genes"
    ),

    "file_selection_performed": False,

    "limitations": [
        "Count-based QC is not sequencing-level QC.",
        "Assignment fraction is not alignment rate.",
        "CPM correlations are descriptive.",
        "Different aliquots can contain biological variation.",
        "Gene identifier consistency does not guarantee identical workflow parameters.",
        "No automatic file exclusion performed."
    ]
}

provenance_path = (
    METADATA_DIR /
    "tcga_duplicate_extended_qc_provenance.json"
)

with open(provenance_path, "w") as f:

    json.dump(
        provenance,
        f,
        indent=4
    )

# ------------------------------------------------------------
# 12. Final summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 1.13 SUMMARY")
print("=" * 65)

print("\nFiles assessed:", len(audit))

print(
    "Gene identifier consistency:",
    "PASSED"
    if len(unique_signatures) == 1
    else "REQUIRES REVIEW"
)

print("\nSaved outputs:")

print("1. tcga_duplicate_rnaseq_extended_qc.csv")

print("2. tcga_duplicate_normalized_correlations.csv")

print("3. tcga_duplicate_gene_identifier_signatures.json")

print("4. tcga_duplicate_extended_qc_provenance.json")

print("\nNo additional molecular files downloaded.")

print("No final file selection performed.")

print("\nPATCH 1.13 COMPLETE")
print("=" * 65)